# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theLeverager1705/Flyrank-starter-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm picking this lane over Lane 1 (signal analysis) or Lane 3 (clustering) because it produces an
actual ranked output someone can act on, not just an observation — that matches the kind of
decision-support/scoring work I want to get good at for quant-style roles, where the deliverable
is "what do I do with limited capacity," not "what did we notice." The starter dataset backs this
up directly (numbers in Section 3): the *majority* of content is flagged `trend_direction == down`,
and once I combine that with visibility/demand/CTR rules, roughly two-thirds of the 30,000 rows
trigger at least one "needs review" reason code. That is far too many pages for any human team to
manually triage — which is exactly the situation a ranking model is supposed to solve (turn "too
many candidates" into "here are the top 50, in order, with reasons"). It also gives me a clean,
falsifiable baseline to beat (the starter pipeline's own rule-based score, already in
`outputs/model_report.md`) rather than starting from nothing, so I can spend the next 7 weeks on
making the ranking *better and more honestly validated*, not on inventing a problem from scratch.
I may still move the label from the starter's current-window proxy (`trend_direction == down`)
toward a genuine future-window label (prior 90 days → next 30 days decline) using the warehouse
release once I reach the data-contract step — see Section 4 for why that distinction matters.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** given limited review capacity, *which content pages should a content
strategist / SEO reviewer look at first this week?* Not "which pages are declining" in the
abstract — "which N pages, ranked, given we can only review ~20-50 this week."

**Who acts, and how:** a human content reviewer (at a FlyRank client, or an internal strategist)
works down a ranked queue. For each page in the queue they see a score, a reason code
(e.g. `declining_with_demand`, `low_ctr_visible_page`, `page_one_decay_risk`), and they decide the
action — refresh, expand, protect, prune, or monitor. The model/score never takes the action
itself; a person always decides. That is why this is decision-support, not automation.

**Cost of a wrong call**, and it cuts both ways because review time is scarce:
- *False positive* (page ranked high but nothing is actually wrong): wastes a reviewer's limited
  hours on a non-problem, and — because capacity is fixed — that hour was taken away from a page
  that really did need attention. The cost compounds through opportunity cost, not just wasted time.
- *False negative* (a real, high-value declining page never surfaces near the top): the page keeps
  losing visibility/clicks silently until someone happens to notice, which for a page with real
  demand (high impressions) is a real, ongoing traffic/revenue cost.

Because capacity is fixed and both error types are costly in different currencies (wasted hours vs.
missed real decline), the right metric is a top-K ranking metric like **precision@50** (of the top
50 the queue surfaces, how many were real problems) rather than plain accuracy — accuracy on a
54%-down dataset is nearly free to get "good" by guessing the majority class, and it says nothing
about whether the *top* of the queue is trustworthy, which is the only part a time-limited reviewer
will ever actually look at.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Numbers computed directly from `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients)
in the code cell below:

1. **54.2% of rows (16,262 / 30,000)** are labeled `trend_direction == "down"` — a majority, but on
   its own this is too blunt to act on (see Section 4 — it's a proxy label, not a future outcome).
2. **65.5% of rows (19,650 / 30,000)** trigger at least one of the starter's five baseline "needs
   review" reason codes (stale-but-visible, declining-with-demand, thin-but-visible, page-one decay
   risk, low-CTR-but-visible). That is the real case for this lane: two-thirds of the inventory
   looks "flagged" under simple rules, which is not a usable to-do list — it's a signal that ranking
   (not a checklist) is what turns this into something a reviewer can actually work through.
3. **23.6% of rows (7,076 / 30,000)** are `page_one_decay_risk` — pages already ranking in the top
   10 (`avg_position` 1–10) that are 180+ days old. These are the highest-stakes candidates: they
   already have real visibility to lose, which is where a wrong "ignore it" call costs the most.

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
n = len(df)
print(f"rows: {n}, unique clients: {df['client_id'].nunique()}")

# 1) how much of the inventory is currently bucketed "declining"
down_share = (df["trend_direction"] == "down").mean()
print(f"1) trend_direction == 'down': {(df['trend_direction']=='down').sum():,} rows "
      f"({down_share*100:.1f}%) -- a majority, but too blunt to act on alone (see Section 4)")

# 2) the starter's five baseline reason-code rules, OR'd together -> candidate review pool
stale = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
thin = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
page_one_decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
low_ctr = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)

any_flag = stale | declining_with_demand | thin | page_one_decay_risk | low_ctr
print(f"2) any baseline 'needs review' flag: {any_flag.sum():,} rows ({any_flag.mean()*100:.1f}%) "
      f"-- too many to hand-review, which is the case for ranking over a checklist")

# 3) the highest-stakes slice: already visible (top 10) but aging -- real visibility to lose
print(f"3) page_one_decay_risk (top-10 position, 180+ days old): "
      f"{page_one_decay_risk.sum():,} rows ({page_one_decay_risk.mean()*100:.1f}%)")


rows: 30000, unique clients: 32
1) trend_direction == 'down': 16,262 rows (54.2%) -- a majority, but too blunt to act on alone (see Section 4)
2) any baseline 'needs review' flag: 19,650 rows (65.5%) -- too many to hand-review, which is the case for ranking over a checklist
3) page_one_decay_risk (top-10 position, 180+ days old): 7,076 rows (23.6%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can claim:**
- *Observed* associations in this anonymized 30,000-row starter slice (32 clients) and, later, in
  the warehouse release — e.g. "pages with X combination of signals were more often flagged
  declining in this data."
- *Directional / decision-support* rankings: "this page scores higher than that one, so review it
  first" — a prioritization, not a guarantee.
- Honest comparison against a transparent baseline rule, with a stated validation design
  (client-holdout, so I'm not just memorizing individual clients).

**What this work will never claim:**
- *Causal proof* that refreshing a page **causes** recovery — I have no experiment (no A/B test, no
  before/after causal design), only observational data. "This page is likely worth reviewing" is
  very different from "editing this page will fix it," and I will not blur that line.
- *Google's algorithm* — I have no access to ranking-factor internals; any pattern I find is a
  correlation in FlyRank's measured signals, not a discovered ranking factor.
- The starter label itself is a **proxy**, not ground truth: `is_declining_label` is derived from
  `trend_direction`, which is computed from `trend_pct` — both of which describe the *current*
  window, not a *future* outcome. That means `trend_direction`/`trend_pct` can never be features
  (they'd leak the label into itself), and it means the starter version of this project is
  measuring "did we correctly re-bucket the current data" rather than "did we predict what happens
  next." A stronger version of this lane, which I intend to build toward using the warehouse
  release, redefines the target as a genuine future-window outcome (prior 90 days of features →
  decline over the *next* 30 days) — that is a materially stronger and more honest claim, and I'm
  flagging now, in Week 1, that the starter proxy is a beginning, not the destination.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.